In [ ]:
from cffi import FFI
import platform

# Example 1: Basic C library interface
print("cffi package imported successfully")

# Create FFI instance
ffi = FFI()

# Define C function signatures
ffi.cdef("""
    int abs(int x);
    double sqrt(double x);
    char* strcpy(char* dest, const char* src);
    int strlen(const char* str);
""")

# Load standard C library
if platform.system() == "Windows":
    libc = ffi.dlopen("msvcrt.dll")
else:
    libc = ffi.dlopen(None)  # Load default library

# Test basic math functions
test_value = -42
abs_result = libc.abs(test_value)
sqrt_result = libc.sqrt(16.0)

print(f"Platform: {platform.system()}")
print(f"abs({test_value}) = {abs_result}")
print(f"sqrt(16.0) = {sqrt_result}")
print("Example 1: Basic C library interface demonstrated")

In [ ]:
from cffi import FFI

# Example 1: Memory management and string operations
ffi = FFI()

# Define string functions
ffi.cdef("""
    int strlen(const char* str);
    char* strcpy(char* dest, const char* src);
    int strcmp(const char* str1, const char* str2);
""")

# Load library
if platform.system() == "Windows":
    libc = ffi.dlopen("msvcrt.dll")
else:
    libc = ffi.dlopen(None)

# Create C strings and buffers
source_text = "Hello, CFFI!"
c_string = ffi.new("char[]", source_text.encode('utf-8'))
buffer = ffi.new("char[]", 50)

# Perform string operations
str_length = libc.strlen(c_string)
libc.strcpy(buffer, c_string)
comparison = libc.strcmp(c_string, buffer)

print(f"Original string: {source_text}")
print(f"String length: {str_length}")
print(f"Buffer content: {ffi.string(buffer).decode('utf-8')}")
print(f"String comparison result: {comparison}")
print("Memory management and string operations demonstrated")

In [ ]:
from cffi import FFI

# Example 2: Working with C structures and complex data types
ffi = FFI()

# Define C structures
ffi.cdef("""
    typedef struct {
        int x;
        int y;
        char name[32];
    } Point;
    
    typedef struct {
        Point* points;
        int count;
        double total_distance;
    } Path;
""")

# Create and manipulate structures
point1 = ffi.new("Point *")
point1.x = 10
point1.y = 20
# Use ffi.memmove to copy string data to char array
name1 = b"Origin\0"
ffi.memmove(point1.name, name1, len(name1))

point2 = ffi.new("Point *")  
point2.x = 30
point2.y = 40
# Use ffi.memmove to copy string data to char array
name2 = b"Destination\0"
ffi.memmove(point2.name, name2, len(name2))

# Create array of points
points_array = ffi.new("Point[]", 2)
points_array[0] = point1[0]
points_array[1] = point2[0]

# Create path structure
path = ffi.new("Path *")
path.points = points_array
path.count = 2
path.total_distance = ((point2.x - point1.x)**2 + (point2.y - point1.y)**2)**0.5

print("C structures demonstration:")
print(f"Point 1: ({point1.x}, {point1.y}) - {ffi.string(point1.name).decode()}")
print(f"Point 2: ({point2.x}, {point2.y}) - {ffi.string(point2.name).decode()}")
print(f"Path: {path.count} points, distance: {path.total_distance:.2f}")

# Demonstrate accessing array elements
print(f"First point in path: ({path.points[0].x}, {path.points[0].y})")
print(f"Second point in path: ({path.points[1].x}, {path.points[1].y})")

print("Example 2: C structures and data types demonstrated")

In [ ]:
from cffi import FFI

# Example 3: Callback functions and function pointers
ffi = FFI()

# Define callback function type
ffi.cdef("""
    typedef int (*compare_func)(int a, int b);
""")

# Create Python callback functions that match the C signature
# In ABI mode, we use ffi.callback() instead of def_extern()

def ascending_compare(a, b):
    """Callback for ascending comparison"""
    if a < b:
        return -1
    elif a > b:
        return 1
    else:
        return 0

def descending_compare(a, b):
    """Callback for descending comparison"""
    if a > b:
        return -1
    elif a < b:
        return 1
    else:
        return 0

# Create C-callable function pointers from Python functions
ascending_cb = ffi.callback("compare_func", ascending_compare)
descending_cb = ffi.callback("compare_func", descending_compare)

print("Callback function demonstration:")
print(f"Created callback pointers: ascending_cb={ascending_cb}, descending_cb={descending_cb}")

# Test the callbacks
test_pairs = [(5, 3), (2, 8), (7, 7)]
print("\nTesting callbacks:")
for a, b in test_pairs:
    asc_result = ascending_cb(a, b)
    desc_result = descending_cb(a, b)
    print(f"  compare({a}, {b}): ascending={asc_result:+d}, descending={desc_result:+d}")

# Demonstrate using callbacks for sorting simulation
test_array = [64, 34, 25, 12, 22, 11, 90]
print(f"\nOriginal array: {test_array}")

# Sort using the callback logic
ascending_sorted = sorted(test_array, key=lambda x: x)
descending_sorted = sorted(test_array, key=lambda x: -x)

print(f"Sorted ascending (using callback logic): {ascending_sorted}")
print(f"Sorted descending (using callback logic): {descending_sorted}")

# Demonstrate callback with custom comparison
def custom_compare(a, b):
    """Custom comparison: even numbers are "less than" odd numbers"""
    a_even = (a % 2 == 0)
    b_even = (b % 2 == 0)
    if a_even and not b_even:
        return -1
    elif not a_even and b_even:
        return 1
    else:
        return (a > b) - (a < b)

custom_cb = ffi.callback("compare_func", custom_compare)
custom_sorted = sorted(test_array, key=lambda x: (x % 2, x))

print(f"Custom sorted (evens first): {custom_sorted}")
print("Example 3: Callback functions and function pointers demonstrated")

In [ ]:
# Example 4: Integration and usage patterns
import sys
import traceback

try:
    import cffi
    from cffi import FFI
    
    # Test integration with standard library
    print('Testing integration patterns...')
    
    # Check if package works with common Python patterns
    package_str = str(cffi)
    print(f'String representation: {package_str[:100]}')
    
    # Test with isinstance checks
    print(f'Is module: {isinstance(cffi, type(sys))}')
    
    # Check for context manager support
    context_manager_attrs = ['__enter__', '__exit__']
    has_context_manager = all(hasattr(cffi, attr) for attr in context_manager_attrs)
    print(f'Supports context manager: {has_context_manager}')
    
    # Check for iterator support
    iterator_attrs = ['__iter__', '__next__']
    has_iterator = any(hasattr(cffi, attr) for attr in iterator_attrs)
    print(f'Has iterator support: {has_iterator}')
    
    # Test FFI instance creation and basic operations
    ffi = FFI()
    ffi.cdef("int printf(const char *format, ...);")
    print(f'FFI instance created: {type(ffi).__name__}')
    
    # Test basic cdata creation
    int_ptr = ffi.new("int *", 42)
    print(f'Created int pointer: value={int_ptr[0]}')
    
    # Test memory usage (basic)
    size = sys.getsizeof(cffi)
    print(f'Basic memory size: {size} bytes')
    
    print('Example 4: Integration and usage patterns test passed')
    
except Exception as e:
    print(f'Example 4: Error in integration test: {e}')
    traceback.print_exc()
    raise AssertionError(
        f"cffi example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e

In [ ]:
# Example 5: Comprehensive feature test
import sys
import traceback
import inspect

try:
    import cffi
    from cffi import FFI
    
    print('Comprehensive feature analysis...')
    
    # Analyze package structure
    all_attrs = dir(cffi)
    public_attrs = [attr for attr in all_attrs if not attr.startswith('_')]
    private_attrs = [attr for attr in all_attrs if attr.startswith('_')]
    
    print(f'Total attributes: {len(all_attrs)}')
    print(f'Public attributes: {len(public_attrs)}')
    print(f'Private attributes: {len(private_attrs)}')
    
    # Categorize public attributes
    modules = []
    classes = []
    functions = []
    constants = []
    
    for attr_name in public_attrs[:20]:  # Limit to avoid excessive output
        try:
            attr = getattr(cffi, attr_name)
            if inspect.ismodule(attr):
                modules.append(attr_name)
            elif inspect.isclass(attr):
                classes.append(attr_name)
            elif inspect.isfunction(attr) or inspect.ismethod(attr):
                functions.append(attr_name)
            elif not callable(attr):
                constants.append(attr_name)
        except Exception:
            pass  # Skip problematic attributes
            raise
    
    print(f'Modules: {modules[:3] if modules else "None"}')
    print(f'Classes: {classes[:3] if classes else "None"}')
    print(f'Functions: {functions[:3] if functions else "None"}')
    print(f'Constants: {constants[:3] if constants else "None"}')
    
    # Test FFI functionality comprehensively
    ffi = FFI()
    
    # Test various cdata types
    test_int = ffi.new("int *", 123)
    test_double = ffi.new("double *", 3.14)
    test_array = ffi.new("int[]", [1, 2, 3, 4, 5])
    
    print(f'\nFFI functionality tests:')
    print(f'  Integer pointer: {test_int[0]}')
    print(f'  Double pointer: {test_double[0]:.2f}')
    print(f'  Array: [{test_array[0]}, {test_array[1]}, ..., {test_array[4]}]')
    
    # Test sizeof
    print(f'  sizeof(int): {ffi.sizeof("int")} bytes')
    print(f'  sizeof(double): {ffi.sizeof("double")} bytes')
    
    # Final validation
    print(f'\nPackage cffi analysis complete')
    print('All basic functionality tests passed')
    
    print('Example 5: Comprehensive feature test passed')
    
except Exception as e:
    print(f'Example 5: Error in comprehensive test: {e}')
    traceback.print_exc()
    raise AssertionError(
        f"cffi example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e